# Qwen3-1.7B Healthcare Assistant
Fine-tuned on `ChatDoctor_HealthCareMagic_112k` using LoRA.
Preprocessing pipeline sourced from group teammate's notebook (COMP8420_Group_L_Healthcare.ipynb).

In [11]:
!pip install torch transformers accelerate peft datasets trl bitsandbytes sacremoses
!pip install --upgrade "torchao>=0.16.0"
!pip install contractions textblob
!python -m textblob.download_corpora lite

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
Finished.


In [12]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import re
import contractions
from textblob import TextBlob
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset, DatasetDict

assert torch.cuda.is_available(), "CUDA not available. Please enable GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU: Tesla T4
VRAM: 14.6 GB


## Preprocessing Pipeline

1. **Punctuation & regex cleaning** — removes noise, protects clinical decimals and abbreviations
2. **Contraction expansion** — expands `don't` → `do not`, `I'm` → `I am` etc.
3. **ClinicalBERT tokenizer normalization** — uses ClinicalBERT vocabulary to normalize medical text

In [ ]:
# Punctuation & Regex Cleaning
def clean_punctuation(text):
    """
    Removes punctuation noise while protecting clinical abbreviations
    and decimal numbers (e.g. 98.6, i.e., e.g., dr.).
    Source: COMP8420_Group_L_Healthcare.ipynb — execute_final_period_logic_v5()
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Lock key clinical abbreviations before any cleaning
    text = re.sub(r'i\.e\.', '_LOCK_IE_', text)
    text = re.sub(r'\bi\.e\b', '_LOCK_IE_', text)
    text = re.sub(r'e\.g\.', '_LOCK_EG_', text)
    text = re.sub(r'\be\.g\b', '_LOCK_EG_', text)
    text = re.sub(r'dr\.', '_LOCK_DR_', text)

    # Strip brackets
    text = re.sub(r'[\(\)]', ' ', text)

    # Un-fuse text/numbers stuck together by removed brackets
    text = re.sub(r'([a-z])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-z])', r'\1 \2', text)

    # Strip question marks
    text = text.replace('?', ' ')

    # Protect clinical decimals (e.g. 98.6 → 98_SAFE_DEC_6)
    text = re.sub(r'(\d+)\.(\d+)', r'\1_SAFE_DEC_\2', text)

    # Collapse ellipses to space
    text = re.sub(r'\.{2,}', ' ', text)

    # Split fused word.word boundaries (e.g. "medicine.it" → "medicine it")
    text = re.sub(r'([a-z0-9])\.([a-z])', r'\1 \2', text)

    # Remove trailing punctuation noise
    text = re.sub(r'[.,;!]+(?=\s|$)', '', text)
    text = re.sub(r'\s\.\s', ' ', text)

    # Restore protected tokens
    text = text.replace('_LOCK_IE_', 'i.e')
    text = text.replace('_LOCK_EG_', 'e.g')
    text = text.replace('_LOCK_DR_', 'dr.')
    text = text.replace('_SAFE_DEC_', '.')

    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Contraction Expansion
def expand_contractions(text):
    """
    Expands English contractions using the contractions library.
    e.g. don't → do not, I'm → I am, won't → will not
    Source: COMP8420_Group_L_Healthcare.ipynb — automated_contraction_expander()
    """
    if not isinstance(text, str):
        return ""
    text = contractions.fix(text)
    return re.sub(r'\s+', ' ', text).strip()

# Load ClinicalBERT tokenizer once — used only for vocabulary normalization
print("Loading ClinicalBERT tokenizer for normalization...")
clinical_tokenizer = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")

# Regional/consumer medical terms to preserve as-is
MEDICAL_SAFETY_SHIELD = {
    'panadol', 'nurofen', 'pooing', 'weezy', 'croupy', 'raspy',
    'rattly', 'uti', 'ild', 'hpylori', 'gyno', 'cause'
}

def autocorrect_typos(text):
    """
    Fixes known common typos using TextBlob, skipping medical safety terms.
    Source: COMP8420_Group_L_Healthcare.ipynb — clean_and_autocorrect_typos()
    """
    words = text.split()
    corrected = []
    for word in words:
        clean_word = re.sub(r'[^a-z0-9]', '', word.lower())
        if clean_word in MEDICAL_SAFETY_SHIELD:
            corrected.append(word)
        elif clean_word in ("immediatly", "afect", "experiencing"):
            corrected.append(str(TextBlob(word).correct().lower()))
        else:
            corrected.append(word)
    return " ".join(corrected)

def normalize_with_clinicalbert(text):
    """
    Passes text through ClinicalBERT tokenizer and decodes back to string.
    This normalizes medical vocabulary using ClinicalBERT's trained vocabulary.
    Source: COMP8420_Group_L_Healthcare.ipynb — pure_huggingface_processor()
    """
    if not isinstance(text, str):
        return ""
    text = expand_contractions(text)
    text = autocorrect_typos(text)
    tokens = clinical_tokenizer.tokenize(text)
    clean_text = clinical_tokenizer.convert_tokens_to_string(tokens)
    clean_text = clean_text.replace(" .", ".").replace(" ,", ",").strip()
    return clean_text


def full_preprocess(text):
    """
    Complete preprocessing pipeline:
    Stage 1: Punctuation & regex cleaning
    Stage 2: Contraction expansion
    Stage 3: ClinicalBERT normalization
    """
    text = clean_punctuation(text)           # Stage 1
    text = normalize_with_clinicalbert(text) # Stage 2 + 3
    return text

print("Preprocessing pipeline ready.")

# Quick sanity check
sample = "I've been having chest pain...it's really bad!!! Dr. Smith said my temp is 98.6"
print(f"\nOriginal : {sample}")
print(f"Processed: {full_preprocess(sample)}")

Loading ClinicalBERT tokenizer for normalization...


## Model Loading & LoRA Configuration

In [ ]:
model_name = "Qwen/Qwen3-1.7B"

qwen_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

# Qwen3-1.7B fits on T4 GPU in FP16 — no quantization needed
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"":0},
    trust_remote_code=True
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # Qwen3 uses o_proj not out_proj
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Load Dataset & Apply Preprocessing

In [ ]:
raw_dataset = load_dataset("xDAN-datasets/ChatDoctor_HealthCareMagic_112k", split="train")
raw_dataset = raw_dataset.select(range(5000))
print(f"Total samples: {len(raw_dataset)}")

system_prompt = (
    "You are an experienced family doctor giving clear, direct medical advice. "
    "Answer in second person. Do not roleplay as a patient."
)

# Patterns compiled once outside the function for efficiency
OPEN_PAT = re.compile(
    r'^(I have gone through[^.]*\.|Thanks for your[^.]*\.|I am Chat[^.]*\.|'
    r'I gone through[^.]*\.|Welcome to[^.]*\.|Hello[^.]*\.|Hi[^.]*\.|'
    r'I can understand[^.]*\.|I understand your[^.]*\.)\s*',
    flags=re.IGNORECASE
)
TRAIL_PAT = re.compile(
    r'\s*(Hope\b.*|Wish\b.*|Take care\b.*|Kindly\b.*|Please (?:consult|advise|help)\b.*|'
    r'Let me know\b.*|Thanks for using\b.*|Chat\s?[Dd]octor\b.*)',
    flags=re.IGNORECASE | re.DOTALL
)

def format_chat(example):
    convs = example["conversations"]
    if len(convs) != 2 or convs[0]["from"] != "human" or convs[1]["from"] != "gpt":
        return {"text": ""}

    user_msg      = convs[0]["value"].strip()
    assistant_msg = convs[1]["value"].strip()

    # Apply teammate preprocessing pipeline to user message
    user_msg = full_preprocess(user_msg)

    # Clean assistant response (greeting + trailing noise)
    assistant_clean = OPEN_PAT.sub("", assistant_msg).strip()
    assistant_clean = TRAIL_PAT.sub("", assistant_clean).strip()
    # Replace collegial "we" with patient-directed "you"
    assistant_clean = re.sub(r'\bwe\b', 'you', assistant_clean, flags=re.IGNORECASE)
    assistant_clean = re.sub(r'\bour\b', 'your', assistant_clean, flags=re.IGNORECASE)
    assistant_clean = assistant_clean[:400]

    # Drop samples where cleaning removed all content
    if len(assistant_clean) < 30:
        return {"text": ""}

    text = f"{system_prompt}\nUser: {user_msg}\nAssistant: {assistant_clean}\n"
    return {"text": text}

print("Applying preprocessing pipeline to dataset (this may take a few minutes)...")
dataset = raw_dataset.map(format_chat, remove_columns=raw_dataset.column_names)
dataset = dataset.filter(lambda x: x["text"] != "")
print(f"After preprocessing & cleaning: {len(dataset)} samples")

# 90/10 train/eval split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

In [ ]:
# Show 3 samples to verify preprocessing is working correctly
print("Sample preprocessed training examples:\n")
for i in range(3):
    print(f"--- Sample {i+1} ---")
    print(train_dataset[i]["text"][:400])
    print()

In [ ]:
def tokenize_fn(examples):
    tokenized = qwen_tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_eval  = eval_dataset.map(tokenize_fn,  batched=True, remove_columns=["text"])

tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_eval.set_format(type="torch",  columns=["input_ids", "attention_mask", "labels"])

print("Tokenization done. Example shape:", tokenized_train[0]["input_ids"].shape)

In [8]:
training_args = TrainingArguments(
    output_dir="./qwen3-1.7b-familydoctor",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_steps=400,
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    warmup_steps=100,
    report_to="none",
    gradient_checkpointing=True,
    optim="adafactor",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=qwen_tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

model.save_pretrained("./qwen3-1.7b-familydoctor-final")
qwen_tokenizer.save_pretrained("./qwen3-1.7b-familydoctor-final")
print("Training complete and model saved.")

Step,Training Loss,Validation Loss
200,2.539213,2.541183
400,2.516711,2.503972
600,2.452015,2.490021
800,2.442477,2.474667
1000,2.411075,2.465989
1200,2.376860,2.464917
1400,2.366291,2.462095


Training complete and model saved.


In [9]:
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()

base_model_name = "Qwen/Qwen3-1.7B"
lora_path = "./qwen3-1.7b-familydoctor-final"

qwen_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map={"":0},
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, lora_path)
model.eval()
print("Model loaded successfully.")

def generate_answer(question, max_new_tokens=150):
    """
    Generate a concise answer using greedy decoding.
    Input question is preprocessed using the same pipeline as training data.
    """
    system_prompt = (
        "You are an experienced family doctor giving clear, direct medical advice. "
        "Answer in second person directly to the patient. "
        "Do not start with greetings. Do not roleplay as a patient."
    )

    # Apply same preprocessing to question at inference time
    clean_question = full_preprocess(question)
    prompt = f"{system_prompt}\nUser: {clean_question}\nAssistant:"

    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
            early_stopping=True,
            pad_token_id=qwen_tokenizer.pad_token_id,
            eos_token_id=qwen_tokenizer.eos_token_id,
        )

    full_output = qwen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Assistant:" in full_output:
        assistant_part = full_output.split("Assistant:")[-1]
    else:
        assistant_part = full_output

    # Post-process: strip any remaining greeting patterns
    assistant_part = OPEN_PAT.sub("", assistant_part.strip())
    assistant_part = TRAIL_PAT.sub("", assistant_part).strip()

    return assistant_part.strip()

# Test
test_questions = [
    "What are the early symptoms of a heart attack?",
    "How can I treat a minor burn at home?",
    "What is the difference between Type 1 and Type 2 diabetes?",
    "Is it safe to take ibuprofen during pregnancy?",
    "How often should adults get a flu shot?"
]

print("\n" + "="*80)
print("TESTING FINETUNED QWEN3-1.7B HEALTHCARE ASSISTANT")
print("="*80)

for i, q in enumerate(test_questions, 1):
    print(f"\n[{i}] Question: {q}")
    answer = generate_answer(q, max_new_tokens=120)
    print(f"Answer: {answer}")
    print("-"*80)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k', 'early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded successfully.

TESTING FINETUNED QWEN3-1.7B HEALTHCARE ASSISTANT

[1] Question: What are the early symptoms of a heart attack?
Answer: In my opinion you should first rule out other causes like lung infection or viral illness etc., because these may mimic coronary artery diseases (CAD). So get done ECG 24 hours monitoring if needed. Get routine blood tests including LFT, renal function test, fasting serum lipids profile before starting any medication. If all t
Thank You,
I am very sorry that this has happened but it is important to know how serious things have become so we will be able to help prevent further complications from occurring. First off, do NOT panic
--------------------------------------------------------------------------------

[2] Question: How can I treat a minor burn at home?
Answer: You should apply antibiotic ointment such as Neosporin over all affected areas once daily until healing occurs then switch out with another product if needed but do NOT use an

In [10]:
from google.colab import drive
drive.mount('/content/drive')

# Define the path in Google Drive
gdrive_path = "/content/drive/MyDrive/qwen3_1_7b_familydoctor_final"

# Create the directory if it doesn't exist
import os
os.makedirs(gdrive_path, exist_ok=True)

# Save the model and tokenizer to Google Drive
model.save_pretrained(gdrive_path)
qwen_tokenizer.save_pretrained(gdrive_path)

print(f"Model and tokenizer saved to {gdrive_path}")

Mounted at /content/drive
Model and tokenizer saved to /content/drive/MyDrive/qwen3_1_7b_familydoctor_final
